# Example 7.1.1 / 1: Compute the Root Gap on a Small Model

Companion notebook for **7.1.1 Relaxations, Bounds and Optimality Gaps** in the AMPL lab Documentation, section "Compute the Root Gap on a Small Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Compute the Root Gap on a Small Model

The batch model from Integer Variables (AMPL lab documentation, section 7.1) maximizes profit. The option
`relax_integrality` solves its relaxation; for the integer solve, the HiGHS options `mip:bestbound=1` returns the final best
bound in the suffix `.bestbound`, and `mip:return_gap=3` returns the absolute and relative gaps
in `.absmipgap` and `.relmipgap`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products made in whole batches
param profit {PROD};              # profit per batch
param hours {PROD} > 0;           # hours needed per batch
param available >= 0;             # total hours available
var Make {PROD} integer >= 0;     # number of batches of each product
maximize TotalProfit: sum {p in PROD} profit[p] * Make[p];  # profit of all batches
subject to Capacity: sum {p in PROD} hours[p] * Make[p] <= available;  # hours available

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["profit"] = {"bands": 25, "coils": 32}
ampl.param["hours"] = {"bands": 2, "coils": 3}
ampl.param["available"] = 17

In [5]:
# run.py
ampl.option["solver_msg"] = 0
ampl.option["relax_integrality"] = 1
ampl.solve(solver="highs")
root = ampl.obj["TotalProfit"].value()
ampl.option["relax_integrality"] = 0
ampl.solve(solver="highs", highs_options="mip:bestbound=1 mip:return_gap=3")
best = ampl.obj["TotalProfit"].value()
print("root bound", root, "| incumbent", best)
print("root gap", root - best, "| relative", round((root - best) / best, 4))
print("final bound", ampl.get_value("TotalProfit.bestbound"),
      "| final gap", ampl.get_value("TotalProfit.absmipgap"))

HiGHS 1.14.0:   mip:bestbound = 1
  mip:return_gap = 3

suffix absmipgap OUT;
suffix bestbound OUT;
suffix relmipgap OUT;
root bound 212.5 | incumbent 207.0
root gap 5.5 | relative 0.0266
final bound 207 | final gap 0


### Pure AMPL: data.dat and model.run

The relaxation bound is 212.5 and the integer optimum is 207. Before any branching, the solver
knows only that the optimum lies between the first incumbent it finds and 212.5; if that
incumbent were already 207, the root gap would be 5.5, or 5.5 / 207 = 0.0266 relative to the
incumbent. Branching then lowers the bound to 207, the final gap is 0, and 207 is proved optimal.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param profit := bands 25 coils 32;
param hours := bands 2 coils 3;
param available := 17;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
suffix bestbound OUT;  # best bound returned by HiGHS
suffix absmipgap OUT;  # absolute gap, incumbent to bound
suffix relmipgap OUT;  # relative gap
param root;            # objective of the relaxation
option relax_integrality 1;
solve;
let root := TotalProfit;
option relax_integrality 0;
option highs_options 'mip:bestbound=1 mip:return_gap=3';
solve;
printf "root bound %g | incumbent %g\n", root, TotalProfit;
printf "final bound %g | final gap %g\n", TotalProfit.bestbound, TotalProfit.absmipgap;

HiGHS 1.14.0:   mip:bestbound = 1
  mip:return_gap = 3
root bound 212.5 | incumbent 207
final bound 207 | final gap 0
